# Проверка расчета критериев

Этот notebook проверяет расчет `C1-C4` для выбранного года.

In [ ]:
from importlib import import_module
from pathlib import Path
import sys

# Определяем корень проекта.
project_root = Path.cwd()
if not (project_root / "data" / "trade.xlsx").exists():
    project_root = project_root.parent

# Добавляем корень проекта в пути импорта.
sys.path.insert(0, str(project_root))

loader = import_module("src.1_data_loader.loader")
preprocessing = import_module("src.2_preprocessing.preprocessing")
indicators = import_module("src.3_indicators.indicators")

In [ ]:
# Год расчета выбирается пользователем.
calculation_year = 2025

raw_data = loader.load_trade_data(project_root / "data" / "trade.xlsx")
prepared_data = preprocessing.preprocess_trade_data(raw_data)
yearly_trade = preprocessing.make_yearly_trade_table(prepared_data)
country_import = preprocessing.make_country_import_table(prepared_data)

indicator_values = indicators.calculate_indicators(
    yearly_trade,
    country_import,
    calculation_year,
)

indicator_values.head(20)

In [ ]:
# Проверяем базовые свойства результата.
expected_columns = ["TNVED", "Year", "Import", "Export", "C1", "C2", "C3", "C4"]

assert list(indicator_values.columns) == expected_columns
assert set(indicator_values["Year"].unique()) == {calculation_year}
assert indicator_values["Import"].gt(0).all()
assert indicator_values["C1"].between(0, 1).all()
assert indicator_values["C2"].ge(0).all()
assert indicator_values["C3"].between(0, 1).all()
assert indicator_values["C4"].between(0, 1).all()

print("Rows:", len(indicator_values))
print("C1 sum:", indicator_values["C1"].sum())
indicator_values.describe()